# Project 3 — Connect-4 RL Infrastructure

This notebook contains the shared infrastructure for the Connect-4 reinforcement learning project:

1. **Setup** — imports
2. **Game Environment** — Connect-4 board logic, win checking, encoding helpers
3. **Model Utilities** — loads any teammate's `.keras` / `.h5` model, `get_move()` with all selection modes
4. **Evaluation** — `Agent` interface, head-to-head matches, round-robin tournaments
5. **Sanity Check** — random vs random to confirm everything works

Run all cells in order. After this, the Policy Gradient and DQN training code can use `get_move()`, `ModelAgent`, `play_match`, etc. directly.


## 1. Setup

In [1]:
import numpy as np
import tensorflow as tf

print("numpy:", np.__version__)
print("tensorflow:", tf.__version__)


/Users/jembleton6/miniconda3/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


numpy: 2.3.4
tensorflow: 2.20.0


## 2. Connect-4 Game Environment

**Board convention:**
- Shape: `(6, 7)` numpy array, dtype int8
- Values: `0 = empty`, `1 = player 1`, `-1 = player 2`
- Row 0 is the **TOP** of the board, row 5 is the **BOTTOM**
- Pieces fall to the lowest empty row in their column (gravity)

**Model input convention (2-channel):**
- Shape: `(6, 7, 2)` float32
- Channel 0 = current player's pieces
- Channel 1 = opponent's pieces
- Always from the perspective of the player about to move


In [2]:
# ----- Constants -----
ROWS = 6
COLS = 7
EMPTY = 0
P1 = 1
P2 = -1


# ----- Core board operations -----

def new_board():
    """Return a fresh empty board."""
    return np.zeros((ROWS, COLS), dtype=np.int8)


def legal_moves(board):
    """Return list of column indices where a piece can be dropped."""
    return [c for c in range(COLS) if board[0, c] == EMPTY]


def is_legal(board, col):
    """Check if a column is a legal move."""
    return 0 <= col < COLS and board[0, col] == EMPTY


def drop_piece(board, col, player):
    """
    Drop a piece for `player` into `col`. Returns a NEW board (does not mutate).
    Raises ValueError if column is full or invalid.
    """
    if not is_legal(board, col):
        raise ValueError(f"Illegal move: column {col} is full or invalid")
    new = board.copy()
    for row in range(ROWS - 1, -1, -1):
        if new[row, col] == EMPTY:
            new[row, col] = player
            return new
    raise ValueError(f"Column {col} is full")


def check_win(board, player):
    """Check if `player` has 4 in a row anywhere on the board."""
    # Horizontal
    for r in range(ROWS):
        for c in range(COLS - 3):
            if all(board[r, c + i] == player for i in range(4)):
                return True
    # Vertical
    for r in range(ROWS - 3):
        for c in range(COLS):
            if all(board[r + i, c] == player for i in range(4)):
                return True
    # Diagonal down-right (\\)
    for r in range(ROWS - 3):
        for c in range(COLS - 3):
            if all(board[r + i, c + i] == player for i in range(4)):
                return True
    # Diagonal up-right (/)
    for r in range(3, ROWS):
        for c in range(COLS - 3):
            if all(board[r - i, c + i] == player for i in range(4)):
                return True
    return False


def is_draw(board):
    """Board is a draw if it's full and nobody won. Caller should check win first."""
    return not any(board[0, c] == EMPTY for c in range(COLS))


def game_status(board):
    """
    Returns:
        1  if P1 won
        -1 if P2 won
        0  if draw
        None if game is ongoing
    """
    if check_win(board, P1):
        return 1
    if check_win(board, P2):
        return -1
    if is_draw(board):
        return 0
    return None


# ----- Model input encoding -----

def board_to_2ch(board, current_player):
    """
    Convert (6,7) board to (6,7,2) input for the model, from the perspective
    of `current_player`. Channel 0: my pieces. Channel 1: opponent's pieces.
    """
    mine = (board == current_player).astype(np.float32)
    theirs = (board == -current_player).astype(np.float32)
    return np.stack([mine, theirs], axis=-1)


def board_to_seq(board, current_player):
    """Convert (6,7) board to (42,2) sequence input for transformer models."""
    return board_to_2ch(board, current_player).reshape(42, 2)


# ----- Tactical helpers (for making opponents stronger) -----

def find_winning_move(board, player):
    """Return a column where `player` can win immediately, or None."""
    for col in legal_moves(board):
        if check_win(drop_piece(board, col, player), player):
            return col
    return None


def find_blocking_move(board, player):
    """
    Return a column where `player` must play to block the opponent's immediate
    win, or None.
    """
    opp = -player
    for col in legal_moves(board):
        if check_win(drop_piece(board, col, opp), opp):
            return col
    return None


def non_losing_moves(board, player):
    """
    Return list of legal moves that do NOT immediately hand the opponent a win
    on their next turn. Falls back to all legal moves if every option loses.
    """
    safe = []
    for col in legal_moves(board):
        after_mine = drop_piece(board, col, player)
        if check_win(after_mine, player):
            safe.append(col)
            continue
        opp = -player
        opp_wins = False
        for opp_col in legal_moves(after_mine):
            if check_win(drop_piece(after_mine, opp_col, opp), opp):
                opp_wins = True
                break
        if not opp_wins:
            safe.append(col)
    return safe if safe else legal_moves(board)


# ----- Display -----

def render(board):
    """Pretty-print the board for debugging."""
    symbols = {EMPTY: ".", P1: "X", P2: "O"}
    lines = []
    for row in board:
        lines.append(" ".join(symbols[v] for v in row))
    lines.append(" ".join(str(c) for c in range(COLS)))
    return "\n".join(lines)


In [3]:
# Quick sanity test of the game logic
b = new_board()
print("Empty board:")
print(render(b))
print()

# P1 builds a vertical in column 3
for _ in range(3):
    b = drop_piece(b, 3, P1)
    b = drop_piece(b, 4, P2)
print("After 3 moves each:")
print(render(b))
print(f"P1 winning move: {find_winning_move(b, P1)}  (should be 3)")
print(f"P2 blocking move: {find_blocking_move(b, P2)}  (should be 3)")


Empty board:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
0 1 2 3 4 5 6

After 3 moves each:
. . . . . . .
. . . . . . .
. . . . . . .
. . . X O . .
. . . X O . .
. . . X O . .
0 1 2 3 4 5 6
P1 winning move: 3  (should be 3)
P2 blocking move: 3  (should be 3)


## 3. Model Loading & Move Selection

Handles any teammate's Project 1 model by auto-detecting input shape:
- **CNN-style**: `(6, 7, 2)`
- **Transformer-style**: `(42, 2)`

`get_move()` supports several selection modes:
- `"argmax"` — deterministic best legal move (evaluation, tournament)
- `"stochastic"` — sample from model's distribution (PG rollouts)
- `"epsilon"` — epsilon-greedy (DQN behavior policy)

Optional tactical overrides (`force_win`, `block_loss`) make opponents stronger for adversarial training — use these on M2, leave them off for the model being trained.


In [4]:
# ----- Model loading -----

def load_model(path):
    """
    Load a Keras model from .keras or .h5 file.
    Auto-detects input kind and attaches _input_kind and _name_tag attributes.
    """
    model = tf.keras.models.load_model(path, compile=False)

    input_shape = model.input_shape
    if len(input_shape) == 4 and input_shape[1:] == (6, 7, 2):
        model._input_kind = "cnn"
    elif len(input_shape) == 3 and input_shape[1:] == (42, 2):
        model._input_kind = "transformer"
    else:
        raise ValueError(
            f"Unrecognized input shape {input_shape} for model at {path}. "
            f"Expected (None,6,7,2) or (None,42,2)."
        )

    model._name_tag = path.split("/")[-1]
    return model


def encode_board(board, current_player, model):
    """Encode a board for a specific model based on its input kind."""
    if model._input_kind == "cnn":
        x = board_to_2ch(board, current_player)
    else:
        x = board_to_seq(board, current_player)
    return x[np.newaxis, ...]


# ----- Raw model inference -----

def predict_probs(board, current_player, model):
    """Get raw probability distribution over 7 columns. Does NOT mask illegal."""
    x = encode_board(board, current_player, model)
    probs = model(x, training=False).numpy()[0]
    return probs


def mask_illegal(probs, board):
    """Zero out probs for illegal columns and renormalize."""
    legal = legal_moves(board)
    masked = np.zeros(COLS, dtype=np.float32)
    for c in legal:
        masked[c] = probs[c]
    s = masked.sum()
    if s > 0:
        return masked / s
    uniform = np.zeros(COLS, dtype=np.float32)
    for c in legal:
        uniform[c] = 1.0 / len(legal)
    return uniform


# ----- Move selection -----

def get_move(
    board,
    current_player,
    model,
    mode="stochastic",
    epsilon=0.1,
    uniform_mix=0.0,
    force_win=False,
    block_loss=False,
    rng=None,
):
    """
    Pick a column for `current_player` using `model`.

    Parameters
    ----------
    mode : "argmax", "stochastic", or "epsilon"
    epsilon : exploration rate for "epsilon" mode
    uniform_mix : [0,1]. Blends uniform with model probs for exploration.
    force_win : play immediately winning moves (use for strong opponents)
    block_loss : block immediate losses (use for strong opponents)

    Returns
    -------
    col : int, the chosen column
    probs : (7,) masked probability distribution used for selection
    """
    if rng is None:
        rng = np.random.default_rng()

    # Tactical overrides first
    if force_win:
        w = find_winning_move(board, current_player)
        if w is not None:
            probs = np.zeros(COLS, dtype=np.float32)
            probs[w] = 1.0
            return w, probs
    if block_loss:
        b = find_blocking_move(board, current_player)
        if b is not None:
            probs = np.zeros(COLS, dtype=np.float32)
            probs[b] = 1.0
            return b, probs

    raw = predict_probs(board, current_player, model)
    probs = mask_illegal(raw, board)

    if uniform_mix > 0.0:
        legal = legal_moves(board)
        u = np.zeros(COLS, dtype=np.float32)
        for c in legal:
            u[c] = 1.0 / len(legal)
        probs = (1.0 - uniform_mix) * probs + uniform_mix * u
        probs = probs / probs.sum()

    if mode == "argmax":
        col = int(np.argmax(probs))
    elif mode == "stochastic":
        col = int(rng.choice(COLS, p=probs))
    elif mode == "epsilon":
        if rng.random() < epsilon:
            legal = legal_moves(board)
            col = int(rng.choice(legal))
        else:
            col = int(np.argmax(probs))
    else:
        raise ValueError(f"Unknown mode: {mode}")

    return col, probs


## 4. Evaluation & Match Runner

The `Agent` interface wraps "anything that picks a move" — trained models, random bots, future DQN agents — so `play_match` treats them all identically.

Use `play_match(agent_a, agent_b, n_games=50)` to run a match that alternates who goes first (matches the tournament spec: one game P1, one game P2).


In [5]:
# ----- Agent interface -----

class Agent:
    """Uniform interface for anything that picks a Connect-4 move."""
    name = "Agent"

    def choose(self, board, current_player, rng):
        raise NotImplementedError


class ModelAgent(Agent):
    """Wraps a loaded Keras model + selection mode into an Agent."""

    def __init__(
        self,
        model,
        mode="argmax",
        epsilon=0.0,
        uniform_mix=0.0,
        force_win=False,
        block_loss=False,
        name=None,
    ):
        self.model = model
        self.mode = mode
        self.epsilon = epsilon
        self.uniform_mix = uniform_mix
        self.force_win = force_win
        self.block_loss = block_loss
        self.name = name or getattr(model, "_name_tag", "model")

    def choose(self, board, current_player, rng):
        col, _ = get_move(
            board,
            current_player,
            self.model,
            mode=self.mode,
            epsilon=self.epsilon,
            uniform_mix=self.uniform_mix,
            force_win=self.force_win,
            block_loss=self.block_loss,
            rng=rng,
        )
        return col


class RandomAgent(Agent):
    """Picks uniformly from legal moves. Useful as a sanity baseline."""
    name = "random"

    def choose(self, board, current_player, rng):
        legal = legal_moves(board)
        return int(rng.choice(legal))


# ----- Single game -----

def play_game(agent_p1, agent_p2, rng=None, max_moves=42, record_boards=False):
    """
    Play one game. agent_p1 plays as P1 (moves first), agent_p2 plays as P2.
    Returns dict with result, num_moves, boards (if recorded), moves.
    """
    if rng is None:
        rng = np.random.default_rng()

    board = new_board()
    current = P1
    agents = {P1: agent_p1, P2: agent_p2}
    boards = [board.copy()] if record_boards else None
    moves = []

    for _ in range(max_moves):
        agent = agents[current]
        col = agent.choose(board, current, rng)
        board = drop_piece(board, col, current)
        moves.append((current, col))
        if record_boards:
            boards.append(board.copy())

        status = game_status(board)
        if status is not None:
            return {
                "result": status,
                "num_moves": len(moves),
                "boards": boards,
                "moves": moves,
            }
        current = -current

    return {"result": 0, "num_moves": len(moves), "boards": boards, "moves": moves}


# ----- Match -----

def play_match(agent_a, agent_b, n_games=50, alternate=True, rng=None, verbose=False):
    """
    Play n_games between agent_a and agent_b, alternating who goes first.
    Returns summary dict with wins, draws, avg game length, etc.
    """
    if rng is None:
        rng = np.random.default_rng()

    a_wins = 0
    b_wins = 0
    draws = 0
    total_moves = 0
    a_wins_as_p1 = 0
    a_wins_as_p2 = 0

    for g in range(n_games):
        a_is_p1 = (g % 2 == 0) if alternate else True

        if a_is_p1:
            result = play_game(agent_a, agent_b, rng=rng)
            if result["result"] == 1:
                a_wins += 1
                a_wins_as_p1 += 1
            elif result["result"] == -1:
                b_wins += 1
            else:
                draws += 1
        else:
            result = play_game(agent_b, agent_a, rng=rng)
            if result["result"] == -1:
                a_wins += 1
                a_wins_as_p2 += 1
            elif result["result"] == 1:
                b_wins += 1
            else:
                draws += 1

        total_moves += result["num_moves"]
        if verbose and (g + 1) % max(1, n_games // 10) == 0:
            print(f"  Game {g+1}/{n_games}: {agent_a.name} {a_wins}-{b_wins}-{draws} {agent_b.name}")

    return {
        "agent_a": agent_a.name,
        "agent_b": agent_b.name,
        "n_games": n_games,
        "a_wins": a_wins,
        "b_wins": b_wins,
        "draws": draws,
        "a_win_rate": a_wins / n_games,
        "b_win_rate": b_wins / n_games,
        "draw_rate": draws / n_games,
        "avg_moves": total_moves / n_games,
        "a_wins_as_p1": a_wins_as_p1,
        "a_wins_as_p2": a_wins_as_p2,
    }


def format_match_summary(summary):
    """Pretty-print a match summary dict."""
    a, b = summary["agent_a"], summary["agent_b"]
    lines = [
        f"{a} vs {b}  ({summary['n_games']} games)",
        f"  {a}: {summary['a_wins']} wins ({summary['a_win_rate']*100:.1f}%)",
        f"  {b}: {summary['b_wins']} wins ({summary['b_win_rate']*100:.1f}%)",
        f"  Draws: {summary['draws']} ({summary['draw_rate']*100:.1f}%)",
        f"  Avg game length: {summary['avg_moves']:.1f} moves",
        f"  {a} as P1: {summary['a_wins_as_p1']} wins | as P2: {summary['a_wins_as_p2']} wins",
    ]
    return "\n".join(lines)


# ----- Round-robin -----

def round_robin(agents, n_games_per_pair=20, rng=None, verbose=False):
    """
    Play every agent vs every other agent. Returns dict with summaries, matrix,
    and agent_names (matrix[i,j] = agent i's win rate vs agent j).
    """
    if rng is None:
        rng = np.random.default_rng()

    n = len(agents)
    results = {}
    win_rate_matrix = np.zeros((n, n))

    for i, a in enumerate(agents):
        for j, b in enumerate(agents):
            if i == j:
                win_rate_matrix[i, j] = np.nan
                continue
            if verbose:
                print(f"\n{a.name} vs {b.name}")
            summary = play_match(a, b, n_games=n_games_per_pair, rng=rng, verbose=False)
            results[(a.name, b.name)] = summary
            win_rate_matrix[i, j] = summary["a_win_rate"]

    return {
        "summaries": results,
        "matrix": win_rate_matrix,
        "agent_names": [a.name for a in agents],
    }


## 5. Sanity Check

Random vs random should be roughly 50/50 with a slight P1 edge (first-move advantage) and average game length ~20-25 moves.


In [6]:
rng = np.random.default_rng(42)
r1 = RandomAgent(); r1.name = "random_A"
r2 = RandomAgent(); r2.name = "random_B"

summary = play_match(r1, r2, n_games=200, rng=rng)
print(format_match_summary(summary))


random_A vs random_B  (200 games)
  random_A: 89 wins (44.5%)
  random_B: 110 wins (55.0%)
  Draws: 1 (0.5%)
  Avg game length: 21.3 moves
  random_A as P1: 50 wins | as P2: 39 wins


## 6. Loading Your Project 1 Model (example)

Once you have your `.keras` or `.h5` model file, load it and test it against a random opponent. This is where the real work starts.

**Replace the path below with wherever your model file lives.**


In [8]:
# Example — uncomment and adjust the path when you have your model file ready
#
M1_PATH = "/Users/jembleton6/Downloads/Connect4_Final_Submission/backend/cnn_best.keras"
m1 = load_model(M1_PATH)
print(f"Loaded {m1._name_tag}: input_kind={m1._input_kind}, input_shape={m1.input_shape}")
#
m1_agent = ModelAgent(m1, mode="argmax", name="M1_original")
random_agent = RandomAgent(); random_agent.name = "random"
#
summary = play_match(m1_agent, random_agent, n_games=50, rng=np.random.default_rng(0))
print(format_match_summary(summary))
# A trained CNN should beat random ~95%+ — if not, something is wrong with the model or encoding.


Loaded cnn_best.keras: input_kind=cnn, input_shape=(None, 6, 7, 2)


/Users/jembleton6/miniconda3/lib/python3.13/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_layer_1']
Received: inputs=Tensor(shape=(1, 6, 7, 2))
  warnings.warn(msg)


M1_original vs random  (50 games)
  M1_original: 50 wins (100.0%)
  random: 0 wins (0.0%)
  Draws: 0 (0.0%)
  Avg game length: 9.2 moves
  M1_original as P1: 25 wins | as P2: 25 wins
